In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, grad
from functools import partial
import jax.scipy.linalg
import numpy as np
from scipy.optimize import curve_fit

# =============================================================================
# 1. CORE JAX ENGINE (your existing machinery, lightly cleaned)
# =============================================================================

def get_generators():
    gens = []
    gens.append(jnp.array([[0, 1, 0], [1, 0, 0], [0, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, -1j, 0], [1j, 0, 0], [0, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[1, 0, 0], [0, -1, 0], [0, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, 1], [0, 0, 0], [1, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, -1j], [0, 0, 0], [1j, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, 0], [0, 0, 1], [0, 1, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, 0], [0, 0, -1j], [0, 1j, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[1, 0, 0], [0, 1, 0], [0, 0, -2]], dtype=jnp.complex64) * (0.5/jnp.sqrt(3)))
    return jnp.stack(gens)

GENERATORS = get_generators()

@partial(jit, static_argnums=(1, 2, 3))
def wilson_loop_trace(U_field, origin, R, T):
    L = U_field.shape[0]
    x, y, z, t = origin
    prod = jnp.eye(3, dtype=jnp.complex64)
    # +mu
    for r in range(R):
        prod = prod @ U_field[(x+r) % L, y, z, t, 0]
    # +nu
    for r in range(T):
        prod = prod @ U_field[(x+R) % L, (y+r) % L, z, t, 1]
    # -mu
    for r in range(R):
        prod = prod @ jnp.conjugate(
            jnp.swapaxes(U_field[(x+R-1-r) % L, (y+T) % L, z, t, 0], -1, -2)
        )
    # -nu
    for r in range(T):
        prod = prod @ jnp.conjugate(
            jnp.swapaxes(U_field[x, (y+T-1-r) % L, z, t, 1], -1, -2)
        )
    return jnp.real(jnp.trace(prod)) / 3.0

@jit
def wilson_action(U_field, beta):
    total_trace = 0.0
    for mu in range(4):
        for nu in range(mu + 1, 4):
            U_mu = U_field[:, :, :, :, mu]
            U_nu_shift_mu = jnp.roll(U_field[:, :, :, :, nu], -1, axis=mu)
            U_mu_shift_nu = jnp.roll(U_field[:, :, :, :, mu], -1, axis=nu)
            U_nu = U_field[:, :, :, :, nu]

            U_mu_dag = jnp.conjugate(jnp.swapaxes(U_mu_shift_nu, -1, -2))
            U_nu_dag = jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))

            P = U_mu @ U_nu_shift_mu @ U_mu_dag @ U_nu_dag
            total_trace += jnp.sum(jnp.real(jnp.trace(P, axis1=-2, axis2=-1)))
    return -(beta / 3.0) * total_trace

@jit
def langevin_step(U_field, key, epsilon, beta):
    dS_dU = grad(wilson_action, argnums=0)(U_field, beta)

    U_dag = jnp.conjugate(jnp.swapaxes(U_field, -1, -2))
    force_raw = U_dag @ dS_dU
    force_ah = (force_raw - jnp.conjugate(jnp.swapaxes(force_raw, -1, -2))) / 2.0
    trace_val = jnp.trace(force_ah, axis1=-2, axis2=-1)[..., None, None]
    drift_term = -(force_ah - (trace_val / 3.0) * jnp.eye(3, dtype=jnp.complex64))

    noise_raw = jax.random.normal(key, U_field.shape + (2,))
    noise_c = noise_raw[..., 0] + 1j * noise_raw[..., 1]
    noise_ah = (noise_c - jnp.conjugate(jnp.swapaxes(noise_c, -1, -2))) / 2.0
    trace_noise = jnp.trace(noise_ah, axis1=-2, axis2=-1)[..., None, None]
    noise_traceless = noise_ah - (trace_noise / 3.0) * jnp.eye(3, dtype=jnp.complex64)

    exponent = epsilon * drift_term + jnp.sqrt(epsilon) * noise_traceless
    update_matrix = jax.scipy.linalg.expm(exponent)
    return update_matrix @ U_field

# =============================================================================
# 2. ANALYSIS UTILITIES
# =============================================================================

def compute_integrated_autocorr(series):
    """Standard windowed estimator of integrated autocorrelation time."""
    n = len(series)
    mean = np.mean(series)
    c0 = np.var(series)
    if c0 == 0:
        return 0.0
    tau = 0.5
    max_lag = n // 50
    for t in range(1, max_lag):
        ct = np.mean((series[:-t] - mean) * (series[t:] - mean))
        rho = ct / c0
        if rho <= 0:
            break
        tau += rho
    return tau

def fit_models(betas, taus, sigmas):
    """Weighted LS fits for poly vs exp, with simple AIC-like diagnostics."""
    betas = np.array(betas, dtype=float)
    taus = np.array(taus, dtype=float)
    sigmas = np.array(sigmas, dtype=float)
    weights = 1.0 / (sigmas**2)

    def model_poly(b, a, p):
        return a * (b**p)

    def model_exp(b, a, c):
        return a * np.exp(c * np.sqrt(b))

    # Weighted fits
    popt_poly, _ = curve_fit(
        model_poly, betas, taus, sigma=sigmas, absolute_sigma=True,
        p0=[0.1, 2.0], maxfev=10000
    )
    popt_exp, _ = curve_fit(
        model_exp, betas, taus, sigma=sigmas, absolute_sigma=True,
        p0=[0.01, 3.0], maxfev=10000
    )

    resid_poly = taus - model_poly(betas, *popt_poly)
    resid_exp  = taus - model_exp(betas, *popt_exp)

    chi2_poly = np.sum(weights * resid_poly**2)
    chi2_exp  = np.sum(weights * resid_exp**2)

    dof = len(betas) - 2  # 2 params each
    red_poly = chi2_poly / max(dof, 1)
    red_exp  = chi2_exp  / max(dof, 1)

    # AIC (up to additive constant):
    AIC_poly = 2 * 2 + chi2_poly
    AIC_exp  = 2 * 2 + chi2_exp

    return {
        "poly": dict(params=popt_poly, chi2=chi2_poly, red=red_poly, AIC=AIC_poly),
        "exp":  dict(params=popt_exp,  chi2=chi2_exp,  red=red_exp,  AIC=AIC_exp),
    }

# =============================================================================
# 3. MAIN HIGH-β TEST WITH MULTIPLE CHAINS PER β
# =============================================================================

def run_high_beta_test():
    print("Initializing T13 'High-Beta' test (multi-chain)...")

    L = 8
    betas = [5.8, 6.0, 6.2, 6.4, 6.6]
    epsilon = 0.02

    burn_in   = 2000
    run_length = 40000
    n_chains   = 4

    # Master seed for reproducibility
    master_key = jax.random.PRNGKey(2025)

    # Cold start
    U0 = jnp.broadcast_to(
        jnp.eye(3, dtype=jnp.complex64),
        (L, L, L, L, 4, 3, 3),
    )

    print("\n" + "=" * 90)
    print(f"{'Beta':<8} | {'<W> mean±std':<24} | {'tau mean±std':<24} | {'gap (1/tau)':<18}")
    print("=" * 90)

    beta_list = []
    tau_means = []
    tau_stds  = []

    for i, beta in enumerate(betas):
        beta_key = jax.random.fold_in(master_key, i)

        chain_means = []
        chain_taus  = []

        for c in range(n_chains):
            chain_key = jax.random.fold_in(beta_key, c)
            U = U0

            # Burn-in
            key = chain_key
            for _ in range(burn_in):
                key, sub = jax.random.split(key)
                U = langevin_step(U, sub, epsilon, beta)

            # Measurement trajectory
            w_hist = []
            for _ in range(run_length):
                key, sub = jax.random.split(key)
                U = langevin_step(U, sub, epsilon, beta)
                w = wilson_loop_trace(U, (0, 0, 0, 0), 2, 2)
                w_hist.append(float(w))

            w_hist = np.asarray(w_hist)
            tau = compute_integrated_autocorr(w_hist)

            chain_means.append(np.mean(w_hist))
            chain_taus.append(tau)

        chain_means = np.asarray(chain_means)
        chain_taus  = np.asarray(chain_taus)

        beta_list.append(beta)
        tau_means.append(chain_taus.mean())
        tau_stds.append(chain_taus.std(ddof=1) if n_chains > 1 else 0.0)

        gap_mean = 1.0 / chain_taus.mean() if chain_taus.mean() > 0 else 0.0

        print(
            f"{beta:<8.1f} | "
            f"{chain_means.mean():.4f} ± {chain_means.std(ddof=1):.4f} | "
            f"{chain_taus.mean():.4f} ± {chain_taus.std(ddof=1):.4f} | "
            f"{gap_mean:.4f}"
        )

    print("=" * 90)

    # ---------------- FITTING WITH ERROR BARS ----------------
    print("\n>>> MODEL COMPARISON (weighted fits) <<<")

    tau_means = np.array(tau_means)
    tau_stds  = np.array(tau_stds)
    # Avoid zero sigma
    tau_stds[tau_stds == 0] = tau_means.mean() * 0.05

    fit = fit_models(beta_list, tau_means, tau_stds)
    poly, exp = fit["poly"], fit["exp"]

    print(f"Polynomial: chi2 = {poly['chi2']:.3f}, red = {poly['red']:.3f}, AIC = {poly['AIC']:.3f}")
    print(f"Exponential: chi2 = {exp['chi2']:.3f}, red = {exp['red']:.3f}, AIC = {exp['AIC']:.3f}")

    if exp["AIC"] + 2 < poly["AIC"]:
        print("\nWinner: EXPONENTIAL (tunneling-dominated relaxation).")
    elif poly["AIC"] + 2 < exp["AIC"]:
        print("\nWinner: POLYNOMIAL (convexity-like scaling in this window).")
    else:
        print("\nResult: INCONCLUSIVE in this β-range with current statistics.")

if __name__ == "__main__":
    run_high_beta_test()


Initializing T13 'High-Beta' test (multi-chain)...

Beta     | <W> mean±std             | tau mean±std             | gap (1/tau)       
5.8      | 0.1833 ± 0.0081 | 3.8390 ± 0.0981 | 0.2605
6.0      | 0.1979 ± 0.0046 | 3.7928 ± 0.2469 | 0.2637
6.2      | 0.2063 ± 0.0075 | 3.7878 ± 0.1844 | 0.2640
